# Silver para Gold: Unidades Consumidoras (V-Energy Grid 360)

Versao 1 da Gold. Continua o notebook bronze-to-silver-unidades-consumidoras-v2 e le sempre a Silver persistida.

Tabelas produzidas:

| Tabela | Granularidade | Uso |
|---|---|---|
| gold_dim_unidade_consumidora | 1 linha por UC | Dimensao analitica, sem dados pessoais em claro |
| gold_resumo_ucs_por_municipio | municipio x classe | Visao gerencial (nome mantido por compatibilidade) |
| gold_ucs_por_alimentador | alimentador x subgrupo tarifario x classe | Base para analises de rede (Grid 360) |
| gold_evolucao_ligacoes_por_ano | ano de ligacao x classe | Novas ligacoes e acumulado |
| gold_qualidade_cadastral | 1 linha por metrica | Indicadores de qualidade da Silver e da quarentena |

Este notebook substitui o silver-to-gold-resumo-ucs. Execute as celulas em ordem.

In [0]:
# ==============================================================================
# CELULA 1: Configuracao e leitura da Silver persistida
# ==============================================================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

# Origem
silver_table = "workspace.default.silver_unidades_consumidoras"
quarantine_table = "workspace.default.silver_unidades_consumidoras_quarentena"

# Destinos
gold_dim_uc = "workspace.default.gold_dim_unidade_consumidora"
gold_resumo_municipio = "workspace.default.gold_resumo_ucs_por_municipio"
gold_alimentador = "workspace.default.gold_ucs_por_alimentador"
gold_evolucao = "workspace.default.gold_evolucao_ligacoes_por_ano"
gold_qualidade = "workspace.default.gold_qualidade_cadastral"

NAO_INFORMADO = "Nao informado"

def fmt(n):
    return f"{n:,}".replace(",", ".")

colunas_silver_esperadas = [
    "id_uc", "cpf_cnpj", "tipo_documento", "documento_dv_valido", "nome_titular",
    "logradouro", "numero", "bairro", "municipio", "uf", "cep",
    "classe", "subgrupo_tarifario", "alimentador", "data_ligacao", "status",
    "latitude", "longitude", "ingestion_datetime", "source_file", "silver_processed_datetime",
]

df_silver = spark.table(silver_table)
faltantes = [c for c in colunas_silver_esperadas if c not in df_silver.columns]
assert not faltantes, f"Colunas ausentes na Silver (rode o bronze-to-silver v2): {faltantes}"

total_silver = df_silver.count()
assert total_silver > 0, "Silver vazia: execute o notebook bronze-to-silver antes."
assert df_silver.select("id_uc").distinct().count() == total_silver, "Silver com id_uc duplicado"

print(f"Silver lida: {silver_table} ({fmt(total_silver)} UCs)")

Silver lida: workspace.default.silver_unidades_consumidoras (45.000 UCs)


In [0]:
# ==============================================================================
# CELULA 2: Dimensao de Unidade Consumidora (sem dados pessoais em claro)
# ==============================================================================
# Pseudonimizacao do documento. Com salt em secret scope o hash resiste a ataque de dicionario;
# sem salt (fallback) o CPF e facilmente reidentificavel por forca bruta.
try:
    SALT_DOCUMENTO = dbutils.secrets.get(scope="venergy", key="salt_documento")
except Exception:
    SALT_DOCUMENTO = ""
    print("AVISO: secret venergy/salt_documento nao encontrado. documento_hash sera gerado SEM salt.")

anos_de_ligacao = F.floor(F.months_between(F.current_date(), F.col("data_ligacao")) / 12).cast("int")

df_dim = (
    df_silver
    .withColumn("documento_hash", F.sha2(F.concat(F.lit(SALT_DOCUMENTO), F.col("cpf_cnpj")), 256))
    .withColumn("ano_ligacao", F.year("data_ligacao"))
    .withColumn("anos_de_ligacao", anos_de_ligacao)
    .withColumn(
        "faixa_antiguidade",
        F.when(F.col("data_ligacao").isNull(), F.lit("Sem data"))
        .when(F.col("anos_de_ligacao") < 5, F.lit("0 a 4 anos"))
        .when(F.col("anos_de_ligacao") < 10, F.lit("5 a 9 anos"))
        .when(F.col("anos_de_ligacao") < 20, F.lit("10 a 19 anos"))
        .otherwise(F.lit("20 anos ou mais")),
    )
    .withColumn("tem_coordenada", F.col("latitude").isNotNull() & F.col("longitude").isNotNull())
    .withColumn("gold_processed_datetime", F.current_timestamp())
    .select(
        "id_uc", "documento_hash", "tipo_documento", "documento_dv_valido",
        "municipio", "bairro", "uf", "cep",
        "classe", "subgrupo_tarifario", "alimentador", "status",
        "data_ligacao", "ano_ligacao", "anos_de_ligacao", "faixa_antiguidade",
        "latitude", "longitude", "tem_coordenada",
        "silver_processed_datetime", "gold_processed_datetime",
    )
)
# Nao entram na dimensao: nome_titular, cpf_cnpj, logradouro e numero (dados pessoais).
print("Dimensao definida (lazy).")

AVISO: secret venergy/salt_documento nao encontrado. documento_hash sera gerado SEM salt.
Dimensao definida (lazy).


In [0]:
# ==============================================================================
# CELULA 3: Agregacoes (todas derivadas da dimensao, para manter consistencia)
# ==============================================================================
def nao_informado(c):
    return F.coalesce(F.col(c), F.lit(NAO_INFORMADO))

def soma_se(condicao):
    return F.sum(F.when(condicao, 1).otherwise(0))

# 3.1 Resumo por municipio x classe
df_resumo = (
    df_dim
    .withColumn("municipio", nao_informado("municipio"))
    .withColumn("classe", nao_informado("classe"))
    .groupBy("municipio", "classe")
    .agg(
        F.count(F.lit(1)).alias("total_unidades"),
        soma_se(F.col("tipo_documento") == "CPF").alias("qtd_cpf"),
        soma_se(F.col("tipo_documento") == "CNPJ").alias("qtd_cnpj"),
        soma_se(F.col("tem_coordenada")).alias("qtd_com_coordenada"),
    )
    .withColumn("pct_com_coordenada", F.round(100.0 * F.col("qtd_com_coordenada") / F.col("total_unidades"), 2))
    .withColumn("gold_processed_datetime", F.current_timestamp())
)

# 3.2 Alimentador x subgrupo tarifario x classe
df_alim = (
    df_dim
    .withColumn("alimentador", nao_informado("alimentador"))
    .withColumn("subgrupo_tarifario", nao_informado("subgrupo_tarifario"))
    .withColumn("classe", nao_informado("classe"))
    .groupBy("alimentador", "subgrupo_tarifario", "classe")
    .agg(
        F.count(F.lit(1)).alias("total_unidades"),
        F.countDistinct("municipio").alias("qtd_municipios"),
        soma_se(F.col("tipo_documento") == "CNPJ").alias("qtd_cnpj"),
        soma_se(F.col("tem_coordenada")).alias("qtd_com_coordenada"),
    )
    .withColumn("gold_processed_datetime", F.current_timestamp())
)

# 3.3 Evolucao das ligacoes por ano e classe (UCs sem data_ligacao ficam fora e sao contadas a parte)
n_sem_data = df_dim.filter(F.col("ano_ligacao").isNull()).count()
janela_acum = (
    Window.partitionBy("classe").orderBy("ano_ligacao")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
df_evolucao = (
    df_dim
    .filter(F.col("ano_ligacao").isNotNull())
    .withColumn("classe", nao_informado("classe"))
    .groupBy("ano_ligacao", "classe")
    .agg(F.count(F.lit(1)).alias("novas_ligacoes"))
    .withColumn("acumulado_ligacoes", F.sum("novas_ligacoes").over(janela_acum))
    .withColumn("gold_processed_datetime", F.current_timestamp())
)
print(f"Agregacoes definidas (lazy). UCs sem data_ligacao (fora da evolucao): {fmt(n_sem_data)}")

Agregacoes definidas (lazy). UCs sem data_ligacao (fora da evolucao): 0


In [0]:
# ==============================================================================
# CELULA 4: Indicadores de qualidade cadastral (Silver e quarentena)
# ==============================================================================
q = df_silver.agg(
    soma_se(F.col("data_ligacao").isNull()).alias("sem_data_ligacao"),
    soma_se(F.col("latitude").isNull() | F.col("longitude").isNull()).alias("sem_coordenada"),
    soma_se(F.col("cep").isNull()).alias("sem_cep"),
    soma_se(F.col("uf").isNull()).alias("sem_uf"),
    soma_se(F.col("municipio").isNull()).alias("sem_municipio"),
    soma_se(F.col("classe").isNull()).alias("sem_classe"),
    soma_se(F.col("alimentador").isNull()).alias("sem_alimentador"),
    soma_se(~F.col("documento_dv_valido")).alias("documento_dv_invalido"),
    soma_se(F.col("tipo_documento") == "CNPJ").alias("total_cnpj"),
).collect()[0]

linhas = [("silver", "total_ucs", total_silver, total_silver, 100.0)]
for rotulo, chave in [
    ("sem_data_ligacao", "sem_data_ligacao"), ("sem_coordenada", "sem_coordenada"),
    ("sem_cep", "sem_cep"), ("sem_uf", "sem_uf"), ("sem_municipio", "sem_municipio"),
    ("sem_classe", "sem_classe"), ("sem_alimentador", "sem_alimentador"),
    ("documento_dv_invalido", "documento_dv_invalido"), ("total_cnpj", "total_cnpj"),
]:
    valor = int(q[chave] or 0)
    linhas.append(("silver", rotulo, valor, total_silver, round(100.0 * valor / total_silver, 2)))

try:
    rej = spark.table(quarantine_table).groupBy("motivo_rejeicao").agg(F.count(F.lit(1)).alias("qtd")).collect()
    for r in sorted(rej, key=lambda x: -x["qtd"]):
        linhas.append(("quarentena", f"motivo:{r['motivo_rejeicao']}", int(r["qtd"]), None, None))
except Exception as e:
    print(f"AVISO: quarentena indisponivel, indicadores de rejeicao omitidos: {e}")

schema_qualidade = StructType([
    StructField("categoria", StringType(), False),
    StructField("metrica", StringType(), False),
    StructField("valor", LongType(), False),
    StructField("total_referencia", LongType(), True),
    StructField("percentual", DoubleType(), True),
])
df_qualidade = (
    spark.createDataFrame(linhas, schema_qualidade)
    .withColumn("gold_processed_datetime", F.current_timestamp())
)
print(f"Indicadores montados: {len(linhas)} linhas.")

Indicadores montados: 12 linhas.


In [0]:
# ==============================================================================
# CELULA 5: Gravacao das tabelas Gold (Delta)
# ==============================================================================
def gravar(df, tabela, comentario):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela)
    )
    try:
        spark.sql(f"COMMENT ON TABLE {tabela} IS '{comentario}'")
    except Exception as e:
        print(f"AVISO: comentario nao aplicado em {tabela}: {e}")
    print(f"Gold gravada: {tabela}")

gravar(df_dim, gold_dim_uc, "Dimensao de UC: 1 linha por id_uc, documento pseudonimizado, sem nome e endereco.")
gravar(df_resumo, gold_resumo_municipio, "Resumo de UCs por municipio e classe.")
gravar(df_alim, gold_alimentador, "UCs por alimentador, subgrupo tarifario e classe.")
gravar(df_evolucao, gold_evolucao, "Novas ligacoes e acumulado por ano de ligacao e classe (base atual da Silver).")
gravar(df_qualidade, gold_qualidade, "Indicadores de qualidade cadastral da Silver e da quarentena.")

Gold gravada: workspace.default.gold_dim_unidade_consumidora
Gold gravada: workspace.default.gold_resumo_ucs_por_municipio
Gold gravada: workspace.default.gold_ucs_por_alimentador
Gold gravada: workspace.default.gold_evolucao_ligacoes_por_ano
Gold gravada: workspace.default.gold_qualidade_cadastral


In [0]:
# ==============================================================================
# CELULA 6: Validacoes BLOQUEANTES sobre as tabelas gravadas
# ==============================================================================
falhas = []

def checar(nome, condicao, detalhe=""):
    print(f"[{'OK   ' if condicao else 'FALHA'}] {nome} {detalhe}")
    if not condicao:
        falhas.append(nome)

def chave_unica(df, chaves):
    return df.count() == df.select(*chaves).distinct().count()

dim = spark.table(gold_dim_uc)
resumo = spark.table(gold_resumo_municipio)
alim = spark.table(gold_alimentador)
ev = spark.table(gold_evolucao)
qual = spark.table(gold_qualidade)

# Dimensao
pii = {"nome_titular", "cpf_cnpj", "logradouro", "numero"} & set(dim.columns)
checar("dimensao sem dados pessoais em claro", not pii, f"{sorted(pii)}")
checar("dimensao: uma linha por UC da Silver", dim.count() == total_silver)
checar("dimensao: id_uc unico", chave_unica(dim, ["id_uc"]))
checar("dimensao: documento_hash preenchido (64 caracteres)",
       dim.filter(F.col("documento_hash").isNull() | (F.length("documento_hash") != 64)).count() == 0)

# Resumo por municipio x classe
s = resumo.agg(F.sum("total_unidades"), F.sum("qtd_cpf"), F.sum("qtd_cnpj")).collect()[0]
checar("resumo: soma de unidades = Silver", s[0] == total_silver, f"({s[0]} x {total_silver})")
checar("resumo: CPF + CNPJ = total", (s[1] or 0) + (s[2] or 0) == total_silver)
checar("resumo: chave (municipio, classe) unica", chave_unica(resumo, ["municipio", "classe"]))

# Alimentador
checar("alimentador: soma de unidades = Silver", alim.agg(F.sum("total_unidades")).collect()[0][0] == total_silver)
checar("alimentador: chave unica", chave_unica(alim, ["alimentador", "subgrupo_tarifario", "classe"]))

# Evolucao
soma_ev = ev.agg(F.sum("novas_ligacoes")).collect()[0][0] or 0
checar("evolucao: novas ligacoes + sem data = Silver", soma_ev + n_sem_data == total_silver, f"({soma_ev} + {n_sem_data})")
checar("evolucao: chave (ano, classe) unica", chave_unica(ev, ["ano_ligacao", "classe"]))
incoerentes = (
    ev.groupBy("classe")
    .agg(F.sum("novas_ligacoes").alias("soma"), F.max("acumulado_ligacoes").alias("maximo"))
    .filter(F.col("soma") != F.col("maximo"))
    .count()
)
checar("evolucao: acumulado final = soma por classe", incoerentes == 0)

# Qualidade
total_q = qual.filter((F.col("categoria") == "silver") & (F.col("metrica") == "total_ucs")).collect()[0]["valor"]
checar("qualidade: total_ucs = Silver", total_q == total_silver)

assert not falhas, f"Validacoes bloqueantes falharam: {falhas}"
print("\nTodas as validacoes bloqueantes da Gold passaram.")

[OK   ] dimensao sem dados pessoais em claro []
[OK   ] dimensao: uma linha por UC da Silver 
[OK   ] dimensao: id_uc unico 
[OK   ] dimensao: documento_hash preenchido (64 caracteres) 
[OK   ] resumo: soma de unidades = Silver (45000 x 45000)
[OK   ] resumo: CPF + CNPJ = total 
[OK   ] resumo: chave (municipio, classe) unica 
[OK   ] alimentador: soma de unidades = Silver 
[OK   ] alimentador: chave unica 
[OK   ] evolucao: novas ligacoes + sem data = Silver (45000 + 0)
[OK   ] evolucao: chave (ano, classe) unica 
[OK   ] evolucao: acumulado final = soma por classe 
[OK   ] qualidade: total_ucs = Silver 

Todas as validacoes bloqueantes da Gold passaram.


In [0]:
# ==============================================================================
# CELULA 7: Amostras e alertas (nao bloqueiam)
# ==============================================================================
print("=== Indicadores de qualidade ===")
display(spark.table(gold_qualidade).orderBy("categoria", F.col("valor").desc()))

print("=== Top 15 municipios por UCs ===")
display(
    spark.table(gold_resumo_municipio)
    .groupBy("municipio").agg(F.sum("total_unidades").alias("total_unidades"))
    .orderBy(F.col("total_unidades").desc()).limit(15)
)

print("=== Top 15 alimentadores por UCs ===")
display(
    spark.table(gold_alimentador)
    .groupBy("alimentador").agg(F.sum("total_unidades").alias("total_unidades"))
    .orderBy(F.col("total_unidades").desc()).limit(15)
)

print("=== Evolucao de ligacoes (primeiras linhas) ===")
display(spark.table(gold_evolucao).orderBy("classe", "ano_ligacao").limit(30))

print("=== Distribuicao por faixa de antiguidade ===")
display(spark.table(gold_dim_uc).groupBy("faixa_antiguidade").count().orderBy("faixa_antiguidade"))

n_nao_inf = spark.table(gold_resumo_municipio).filter(F.col("municipio") == NAO_INFORMADO).agg(F.sum("total_unidades")).collect()[0][0] or 0
if n_nao_inf > 0:
    print(f"ALERTA: {fmt(n_nao_inf)} UCs sem municipio aparecem como '{NAO_INFORMADO}' no resumo.")

=== Indicadores de qualidade ===


categoria,metrica,valor,total_referencia,percentual,gold_processed_datetime
quarentena,motivo:dado_de_teste_id_uc,600,null,null,2026-10-29T19:37:07.538Z
quarentena,motivo:cabecalho_repetido,3,null,null,2026-10-29T19:37:07.538Z
silver,total_ucs,45000,45000,100.0,2026-10-29T19:37:07.538Z
silver,total_cnpj,6008,45000,13.35,2026-10-29T19:37:07.538Z
silver,sem_cep,1350,45000,3.0,2026-10-29T19:37:07.538Z
silver,sem_uf,0,45000,0.0,2026-10-29T19:37:07.538Z
silver,sem_data_ligacao,0,45000,0.0,2026-10-29T19:37:07.538Z
silver,sem_municipio,0,45000,0.0,2026-10-29T19:37:07.538Z
silver,sem_coordenada,0,45000,0.0,2026-10-29T19:37:07.538Z
silver,sem_classe,0,45000,0.0,2026-10-29T19:37:07.538Z


=== Top 15 municipios por UCs ===


municipio,total_unidades
Recife,17995
Jaboatao Dos Guararapes,7800
Olinda,4212
Paulista,4024
Cabo De Santo Agostinho,2513
Camaragibe,1783
Igarassu,1400
Sao Lourenco Da Mata,1398
Ipojuca,1176
Abreu E Lima,1165


=== Top 15 alimentadores por UCs ===


alimentador,total_unidades
ALM-REC-008,2285
ALM-REC-024,1338
ALM-JAB-007,1222
ALM-SAO-004,1091
ALM-REC-004,1012
ALM-PAU-003,982
ALM-JAB-001,979
ALM-OLI-001,936
ALM-JAB-004,863
ALM-PAU-011,692


=== Evolucao de ligacoes (primeiras linhas) ===


ano_ligacao,classe,novas_ligacoes,acumulado_ligacoes,gold_processed_datetime
1904,Comercial,1,1,2026-10-29T19:37:02.941Z
1905,Comercial,1,2,2026-10-29T19:37:02.941Z
1910,Comercial,1,3,2026-10-29T19:37:02.941Z
1912,Comercial,1,4,2026-10-29T19:37:02.941Z
1920,Comercial,1,5,2026-10-29T19:37:02.941Z
1923,Comercial,1,6,2026-10-29T19:37:02.941Z
1924,Comercial,1,7,2026-10-29T19:37:02.941Z
1925,Comercial,1,8,2026-10-29T19:37:02.941Z
1931,Comercial,1,9,2026-10-29T19:37:02.941Z
1937,Comercial,1,10,2026-10-29T19:37:02.941Z


=== Distribuicao por faixa de antiguidade ===


faixa_antiguidade,count
0 a 4 anos,586
10 a 19 anos,18600
20 anos ou mais,16704
5 a 9 anos,9110
